# 5a. Full workflow: Marianas arc data

Here's an example of a full workflow to explore melt inclusion and matrix glass data using VolFe. We'll calculate saturation pressures, degassing paths, isobars, and *f*<sub>O<sub>2</sub></sub> from sulfur content from a single csv file in one notebook. We also compare to some of the other degassing tools available. This example is used in Hughes et al. (pre-print).

## Python set-up
First we need to import a few Python packages (including VolFe). You need to install VolFe (and the other packages such as ThermoBar) once on your machine. If you haven't yet, uncomment the line below (remove the # for the lines beginning !pip).

In [1]:
# Install VolFe and Thermobar on your machine if you have not done so already. Remove the # from lines below to do this (don't remove the # from this line!).
# !pip install VolFe
# !pip install Thermobar
# !pip install VESIcal

In [2]:
# import python packages
import pandas as pd
import numpy as np
import VolFe as vf
import Thermobar as tb
import VESIcal as vc
import plotly.graph_objects as go
from plotly.subplots import make_subplots

c:\Users\ehughes\AppData\Local\miniforge3\envs\volfe-dev\Lib\site-packages\VESIcal\calculate_classes.py:7: UserWarning: 

  from VESIcal.models import magmasat


## Import data

First we will load our data from a csv file. We'll use the examples_marianas csv in files again from Brounce et al. (2014, 2016) and Kelley & Cottrell (2012).

In [3]:
my_analyses = pd.read_csv("../../files/example_marianas.csv")  # load data

## Calculate temperature using Thermobar

All VolFe calculations require a temperature. We calculate temperature for each glass analysis using eq. (14) from Putirka (2008), which depends on melt composition including water but not pressure, using ThermoBar (Wieser et al. 2022). This is then added to the dataframe of compositions.

In [4]:
# renames melt composition headers to be comparible with Thermobar
my_analyses_tb = my_analyses.rename(columns = {"SiO2":"SiO2_Liq","TiO2":"TiO2_Liq","Al2O3":"Al2O3_Liq","FeOT":"FeOt_Liq","MnO":"MnO_Liq","MgO":"MgO_Liq","CaO":"CaO_Liq",
                                               "Na2O":"Na2O_Liq","K2O":"K2O_Liq","P2O5":"P2O5_Liq","H2O":"H2O_Liq"})
# calculates T using Thermobar based on eq. (14) from Putirka (2008) and converts to 'C
T_C_all = tb.calculate_liq_only_temp(liq_comps=my_analyses_tb, equationT="T_Put2008_eq14")-273.15
# adds temperatures to original dataframe
my_analyses.insert(1,"T_C",T_C_all)

## Calculate the pressure of vapor-saturation

Now we can calculate the pressure of vapor saturation for all the analyses in the file using the default models in VolFe.

In [5]:
pvsat = vf.calc_Pvsat(my_analyses)

## Calculate the *f*<sub>O<sub>2</sub></sub> from the sulfur content

Instead of using the Fe<sup>3+</sup>/Fe<sub>T</sub> measured in the glass to calculate *f*<sub>O<sub>2</sub></sub>, we can instead use the measured S content and compare it too the S<sup>2-</sup>CSS.

In [6]:
fO2fromS = vf.calc_melt_S_oxybarometer(my_analyses)

## Influence of melt composition errors

Next we look at how errors in the melt composition might influence the calculations we have done. We use the composition of melt inclusion Ala02-16A. Next we use a Monte Carlo approach to calculate 1000 potential compositions of Ala02-16A to see how that effects our calculations.

In [7]:
comp_error = vf.calc_comp_error(my_analyses,29,1000)

We can work out the temperature associated with each melt composition again using eq. (14) from Putirka (2008) using ThermoBar (Wieser et al. 2022). First we have to change the header and then add the results to the dataframe of compositions.

In [8]:
# removes the first three rows that detail the original composition, the error size, and its type
comp_error = comp_error.drop([1,2,3]) 
# reset the index so the first row is 0 again
comp_error = comp_error .reset_index() 
# remove the current temperature value
comp_error = comp_error.drop(['T_C'], axis=1) 
# rename the columns so it is compatible with Thermobar
comp_error_tb = comp_error.rename(columns = {"SiO2":"SiO2_Liq","TiO2":"TiO2_Liq","Al2O3":"Al2O3_Liq","FeOT":"FeOt_Liq","MnO":"MnO_Liq","MgO":"MgO_Liq","CaO":"CaO_Liq",
                                               "Na2O":"Na2O_Liq","K2O":"K2O_Liq","P2O5":"P2O5_Liq","H2O":"H2O_Liq"})
# calculate the temperature and convert to 'C
T_C_error = tb.calculate_liq_only_temp(liq_comps=comp_error_tb, equationT="T_Put2008_eq14")-273.15
# add the temperatures to the dataframe
comp_error.insert(1,"T_C",T_C_error)

c:\Users\ehughes\AppData\Local\miniforge3\envs\volfe-dev\Lib\site-packages\Thermobar\core.py:474: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  oxide_mass_liq_anhyd_df.columns, axis=1).fillna(0)
c:\Users\ehughes\AppData\Local\miniforge3\envs\volfe-dev\Lib\site-packages\Thermobar\core.py:474: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  oxide_mass_liq_anhyd_df.columns, axis=1).fillna(0)


Next we calculate the *P<sup>v</sup>*<sub>sat</sub> for each composition. After, we calculate the mean and standard deviation for *T*, *P<sup>v</sup>*<sub>sat</sub>, *P<sup>v</sup>*<sub>sat</sub>_diff, and *f*<sub>O<sub>2</sub></sub> using the measured Fe<sup>3+</sup>/Fe<sub>T</sub>.

In [9]:
pvsat_error = vf.calc_Pvsat(comp_error)

In [10]:
pvsat_stats = {"T_mean":np.mean(pvsat_error["T_C"]),"T_sd":np.std(pvsat_error["T_C"]),
               "P_mean":np.mean(pvsat_error["P_bar"]),"P_sd":np.std(pvsat_error["P_bar"]),
               "fO2_mean":np.mean(pvsat_error["fO2_DFMQ"]),"fO2_sd":np.std(pvsat_error["fO2_DFMQ"]),
               "P_diff_mean":np.mean(pvsat_error["Pvsat_diff_bar"]),"P_diff_sd":np.std(pvsat_error["Pvsat_diff_bar"])}

And a similar analysis for *f*<sub>O<sub>2</sub></sub> from measured S.

In [11]:
fO2fromS_error = vf.calc_melt_S_oxybarometer(comp_error)

In [12]:
fO2fromS_stats = {"fO2_mean":np.mean(fO2fromS_error["DFMQ-sulfide"]),"fO2_sd":np.std(fO2fromS_error["DFMQ-sulfide"]),
               "P_mean":np.mean(fO2fromS_error["P (bar) sulf"]),"P_sd":np.std(fO2fromS_error["P (bar) sulf"])}

## Calculate isobars

We can also calculate isobars for Ala02-16A but we need to change the "insolubles" option to "H2O-CO2 only" to do this.

In [13]:
# change just the "COH_species" option to "H2O-CO2 only"
my_models = [['COH_species','H2O-CO2 only']]
# turn "my_models" to dataframe with correct column headers and indexes    
my_models = vf.make_df_and_add_model_defaults(my_models)
# calculate isobars for Ala01-16A which is run 29 in the csv starting at 1000 bars, ending at 4000 bars at 1000 bar intervals
isobars = vf.calc_isobar(my_analyses,run=29,models=my_models,initial_P=1000.,final_P=5000.,step_P=1000.)
# split into each pressure for plotting
isobar1000 = isobars[isobars["P_bar"]==1000.]
isobar2000 = isobars[isobars["P_bar"]==2000.]
isobar3000 = isobars[isobars["P_bar"]==3000.]
isobar4000 = isobars[isobars["P_bar"]==4000.]
isobar5000 = isobars[isobars["P_bar"]==5000.]

## Calculate de/regassing paths

### Closed-system, melt-only, degassing

Next we can run a closed-system degassing calculation starting from Ala02-16A, using the default options.

In [14]:
degassing_closed_1 = vf.calc_gassing(my_analyses,run=29)

1.0 : Switching solve species from OCS to OCH (first time)
1.0 : Switching solve species from OCH to OHS (second time)
1.0 : Switching solve species from OCS to OCH (first time)
1.0 : Switching solve species from OCH to OHS (second time)
solver failed, calculation aborted at P =  1.0 2025-02-26 15:48:36.554141


### Closed-system, melt+vapor, degassing

But it probably had more CO<sub>2</sub> to start with... so we can run a closed-system degassing calculation assuming the initial melt contained 2 wt% CO<sub>2</sub>.

In [15]:
# 'bulk composition' to 'melt+vapor_initialCO2'
my_models = [['bulk_composition',"melt+vapor_initialCO2"]]

# turn to dataframe with correct column headers and indexes    
my_models = vf.make_df_and_add_model_defaults(my_models)

degassing_closed_2 = vf.calc_gassing(my_analyses,run=29,models=my_models)

1.0 : Switching solve species from OCS to OCH (first time)
1.0 : Switching solve species from OCH to OHS (second time)
1.0 : Switching solve species from OCS to OCH (first time)
1.0 : Switching solve species from OCH to OHS (second time)
solver failed, calculation aborted at P =  1.0 2025-02-26 15:49:48.722562


### Closed-system, melt+gas, regassing

... and regas along the same path to see how it would have evolved at higher pressures before the melt inclusion was formed.

In [16]:
# change to regas and melt+vapor_initialCO2
my_models = [['gassing_direction','regas'],['bulk_composition',"melt+vapor_initialCO2"]]

# turn to dataframe with correct column headers and indexes    
my_models = vf.make_df_and_add_model_defaults(my_models)

regassing_closed_2 = vf.calc_gassing(my_analyses,run=29,models=my_models)

### Open-system degassing

And we can compare it to open-system degassing...

In [17]:
# change just the "COH_species" option to "H2O-CO2 only"
my_models = [['gassing_style','open']]

# turn to dataframe with correct column headers and indexes    
my_models = vf.make_df_and_add_model_defaults(my_models)

degassing_open = vf.calc_gassing(my_analyses,run=29,models=my_models)

### Open-system regassing

... and open-system regassing ...

In [18]:
# change just the "COH_species" option to "H2O-CO2 only"
my_models = [['gassing_style','open'],['gassing_direction','regas']]

# turn to dataframe with correct column headers and indexes    
my_models = vf.make_df_and_add_model_defaults(my_models)

regassing_open = vf.calc_gassing(my_analyses,run=29,models=my_models)

## Comparison to other tools

We can compare the results of the closed-system, melt-only, degassing calculation to the results from other tools that are available.

### VESIcal (Iacovino et al., 2021)

Using the VolatileCalc model (Newman & Lowerstern, 2022), which is closest to the model dependent variable options used in the VolFe calculation.

In [19]:
my_sample = vc.Sample({'SiO2':  43.97,
 'TiO2':   0.7,
 'Al2O3': 19.09,
 'Fe2O3':  0.,
 'Cr2O3':  0.0,
 'FeO':    9.36,
 'MnO':    0.22,
 'MgO':    6.76,
 'NiO':    0.0,
 'CoO':    0.0,
 'CaO':    13.28,
 'Na2O':   1.46,
 'K2O':    0.37,
 'P2O5':   0.11,
 'H2O':    4.32,
 'CO2':    0.0999})

vesical_volatilecalc = vc.calculate_degassing_path(sample=my_sample,
                                temperature=1111.,
                                model='Dixon').result

### Sulfur_X (Ding et al., 2023)

Initial conditions and model options can be found in 'main_Fuego.py' and 'melt_composition.py'.

In [20]:
sulfur_X = pd.read_csv("../../files/comparison/sulfur_X.csv")  # load data

### CHOSETTO (REF)

Initial conditions and model options can be found in "chosetto.ctr" and "chosetto.dat". The original output (chosetto.out) was converted to csv, D was replaced with E throughout, and column headers were added.

In [46]:
chosetto = pd.read_csv("../../files/comparison/chosetto.csv")  # load data

## Plotting

Finally, we can plot all everything!

In [47]:
x = (chosetto['FeO_Fe2O3']*2.*159.69)/71.844
chosetto_Fe3_FeT = 1.-(x/(x+1.))

In [143]:
fig = make_subplots(rows=3, cols=3,
                   shared_yaxes = False, shared_xaxes = False,
                   vertical_spacing=0.07, horizontal_spacing=0.07)

data = my_analyses

# TAS
r = 1
c = 1
fig.add_trace(go.Scatter(mode = "markers", x=data["SiO2"], y=data["K2O"]+data["Na2O"],showlegend = False,
                         marker=dict(symbol=data["Symbol"], color=data["Colour"], line_color="black", size=data['Size'], line_width=1)), row = r, col = c)
fig.add_trace(go.Scatter(mode = "markers", x=[data.loc[29,"SiO2"]], y=[data.loc[29,"K2O"]+data.loc[29,"Na2O"]],showlegend = False,
                         error_x=dict(type='percent', value=2*100.*data.loc[29,"SiO2_sd"],visible=True),
                         error_y=dict(type='percent', value=2*100.*(data.loc[29,"Na2O_sd"]**2+data.loc[29,"K2O_sd"]**2.)**0.5,visible=True),
                         marker=dict(symbol=[data.loc[29,"Symbol"]], color=[data.loc[29,"Colour"]], line_color="black", size=[data.loc[29,'Size']], line_width=1)), row = r, col = c)
fig.update_xaxes(range = [40,60], dtick = 5, title = "SiO<sub>2</sub> (wt%)", row = r, col = c)
fig.update_yaxes(range = [0,5], dtick = 1, title = "Na<sub>2</sub>O+K<sub>2</sub>O (wt%)", row = r, col = c)
fig.add_annotation(x=41, y=4.7,text="a",showarrow=False, bordercolor = "black", bgcolor="white",
                   width=15, height=15,font=dict(color="black",size=15),textangle=0,  row = r, col = c)

# H2O-CO2
r = 2
c = 1
fig.add_trace(go.Scatter(mode = "lines", x=isobar1000["H2O_wtpc"], y=isobar1000["CO2_ppm"], line_color = "darkgrey", line_width = 1,showlegend = False, 
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=isobar2000["H2O_wtpc"], y=isobar2000["CO2_ppm"], line_color = "darkgrey", line_width = 1,showlegend = False, 
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=isobar3000["H2O_wtpc"], y=isobar3000["CO2_ppm"], line_color = "darkgrey", line_width = 1, showlegend = False, 
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=isobar4000["H2O_wtpc"], y=isobar4000["CO2_ppm"], line_color = "darkgrey", line_width = 1, showlegend = False, 
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "markers", x=data["H2O"], y=data["CO2ppm"],showlegend = False, 
                         marker=dict(symbol=data["Symbol"], color=data["Colour"], line_color="black", size=data['Size'], line_width=1)), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_1['H2OT-eq_wtpc'], y=degassing_closed_1['CO2T-eq_ppmw'], line_color = "steelblue", line_width = 3, showlegend=False, 
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_2['H2OT-eq_wtpc'], y=degassing_closed_2['CO2T-eq_ppmw'], line_color = "firebrick", line_width = 3, showlegend=False, 
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_closed_2['H2OT-eq_wtpc'], y=regassing_closed_2['CO2T-eq_ppmw'], line_color = "firebrick", line_width = 3, showlegend=False, 
                         line_dash = "dot"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_open['H2OT-eq_wtpc'], y=degassing_open['CO2T-eq_ppmw'], line_color = "steelblue", line_width = 3, showlegend=False, 
                         line_dash = "dash"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_open['H2OT-eq_wtpc'], y=regassing_open['CO2T-eq_ppmw'], line_color = "steelblue", line_width = 3, showlegend=False, 
                         line_dash = "dot"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "markers", x=[data.loc[29,"H2O"]], y=[data.loc[29,"CO2ppm"]],showlegend = False, 
                         error_x=dict(type='constant', value=2.*data.loc[29,"H2O_sd"],visible=True),
                         error_y=dict(type='constant', value=2.*data.loc[29,"CO2ppm_sd"],visible=True),
                         marker=dict(symbol=[data.loc[29,"Symbol"]], color=[data.loc[29,"Colour"]], line_color="black", size=[data.loc[29,'Size']], line_width=1)), row = r, col = c)
fig.update_xaxes(range = [0,6.1], dtick = 1, title = "H<sub>2</sub>O-eq (wt%)", row = r, col = c)
fig.update_yaxes(range = [0,2000], dtick = 500, title = "CO<sub>2</sub>-eq (ppm)", row = r, col = c)
fig.add_annotation(x=0.3, y=1880,text="b",showarrow=False, bordercolor = "black", bgcolor="white",
                   width=15, height=15,font=dict(color="black",size=15),textangle=0,  row = r, col = c)
fig.add_annotation(x=0.4, y=570,text="1 kb",showarrow=False,font=dict(color="grey",size=15),textangle=0,  row = r, col = c)
fig.add_annotation(x=0.2, y=1120,text="2",showarrow=False,font=dict(color="grey",size=15),textangle=0,  row = r, col = c)
fig.add_annotation(x=1.5, y=1690,text="3",showarrow=False,font=dict(color="grey",size=15),textangle=0,  row = r, col = c)
fig.add_annotation(x=3.6, y=1950,text="4",showarrow=False,font=dict(color="grey",size=15),textangle=0,  row = r, col = c)

# Fe3+/FeT-S
r = 2
c = 2
fig.add_trace(go.Scatter(mode = "markers", x=data["Fe3FeT"], y=data["STppm"],showlegend = False, 
                         marker=dict(symbol=data["Symbol"], color=data["Colour"], line_color="black", size=data['Size'], line_width=1)), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_1['Fe3+/FeT'], y=degassing_closed_1['ST_ppmw'], line_color = "steelblue", line_width = 3, showlegend=False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_2['Fe3+/FeT'], y=degassing_closed_2['ST_ppmw'], line_color = "firebrick", line_width = 3, showlegend=False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_closed_2['Fe3+/FeT'], y=regassing_closed_2['ST_ppmw'], line_color = "firebrick", line_width = 3, showlegend=False,
                         line_dash = "dot"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_open['Fe3+/FeT'], y=degassing_open['ST_ppmw'], line_color = "steelblue", line_width = 3, showlegend=False,
                         line_dash = "dash"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_open['Fe3+/FeT'], y=regassing_open['ST_ppmw'], line_color = "steelblue", line_width = 3, showlegend=False,
                         line_dash = "dot"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "markers", x=[data.loc[29,"Fe3FeT"]], y=[data.loc[29,"STppm"]],showlegend = False, 
                         error_y=dict(type='constant', value=2.*data.loc[29,"STppm_sd"],visible=True),
                         error_x=dict(type='constant', value=2.*data.loc[29,"Fe3FeT_sd"],visible=True),
                         marker=dict(symbol=[data.loc[29,"Symbol"]], color=[data.loc[29,"Colour"]], line_color="black", size=[data.loc[29,'Size']], line_width=1)), row = r, col = c)
fig.update_xaxes(range = [0.1,0.3], dtick = 0.05, title = "Fe<sup>3+</sup>/Fe<sub>T</sub>",tickformat = ".2f", row = r, col = c)
fig.update_yaxes(range = [0,2000], dtick = 500, title = "S<sub>T</sub> (ppm)", side='right', row = r, col = c)
fig.add_annotation(x=0.11, y=1880,text="c",showarrow=False, bordercolor = "black", bgcolor="white",
                   width=15, height=15,font=dict(color="black",size=15),textangle=0,  row = r, col = c)

# H2O-S
r = 2
c = 3
fig.add_trace(go.Scatter(mode = "markers", x=data["H2O"], y=data["STppm"],showlegend = False, 
                         marker=dict(symbol=data["Symbol"], color=data["Colour"], line_color="black", size=data['Size'], line_width=1)), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_1['H2OT-eq_wtpc'], y=degassing_closed_1['ST_ppmw'], line_color = "steelblue", line_width = 3, showlegend=False, 
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_2['H2OT-eq_wtpc'], y=degassing_closed_2['ST_ppmw'], line_color = "firebrick", line_width = 3, showlegend=False, 
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_closed_2['H2OT-eq_wtpc'], y=regassing_closed_2['ST_ppmw'], line_color = "firebrick", line_width = 3, showlegend=False, 
                         line_dash = "dot"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_open['H2OT-eq_wtpc'], y=degassing_open['ST_ppmw'], line_color = "steelblue", line_width = 3, showlegend=False, 
                         line_dash = "dash"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_open['H2OT-eq_wtpc'], y=regassing_open['ST_ppmw'], line_color = "steelblue", line_width = 3, showlegend=False, 
                         line_dash = "dot"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "markers", x=[data.loc[29,"H2O"]], y=[data.loc[29,"STppm"]],showlegend = False, 
                         error_y=dict(type='constant', value=2.*data.loc[29,"STppm_sd"],visible=True),
                         error_x=dict(type='constant', value=2.*data.loc[29,"H2O_sd"],visible=True),
                         marker=dict(symbol=[data.loc[29,"Symbol"]], color=[data.loc[29,"Colour"]], line_color="black", size=[data.loc[29,'Size']], line_width=1)), row = r, col = c)
fig.update_xaxes(range = [0,6.1], dtick = 1, title = "H<sub>2</sub>O-eq (wt%)", row = r, col = c)
fig.update_yaxes(range = [0,2000], dtick = 500, title = "S<sub>T</sub> (ppm)",  side="right",row = r, col = c)
fig.add_annotation(x=0.3, y=1880,text="d",showarrow=False, bordercolor = "black", bgcolor="white",
                   width=15, height=15,font=dict(color="black",size=15),textangle=0,  row = r, col = c)

# H2O-CO2
r = 3
c = 1
fig.add_trace(go.Scatter(mode = "markers", x=data["H2O"], y=data["CO2ppm"],showlegend = False, 
                         marker=dict(symbol=data["Symbol"], color=data["Colour"], line_color="black", size=data['Size'], line_width=1)), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_1['H2OT-eq_wtpc'], y=degassing_closed_1['CO2T-eq_ppmw'], line_color = "steelblue", line_width = 3, showlegend=False, 
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=sulfur_X['wH2O_melt'], y=sulfur_X['wCO2_melt'], line_color = "purple", line_width = 3, showlegend=False, 
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=chosetto['wm_H2O']*100., y=chosetto['wm_CO2']*100000., line_color = "green", line_width = 3, showlegend=False, 
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=vesical_volatilecalc['H2O_liq'], y=vesical_volatilecalc['CO2_liq']*10000., line_color = "black", line_width = 3, showlegend=False, 
                         line_dash = "solid"), row = r, col = c)
#fig.add_trace(go.Scatter(mode = "lines", x=evo['H2OT-eq_wtpc'], y=evo['CO2T-eq_ppmw'], line_color = "steelblue", line_width = 3, showlegend=False, 
#                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "markers", x=[data.loc[29,"H2O"]], y=[data.loc[29,"CO2ppm"]],showlegend = False, 
                         error_x=dict(type='constant', value=2.*data.loc[29,"H2O_sd"],visible=True),
                         error_y=dict(type='constant', value=2.*data.loc[29,"CO2ppm_sd"],visible=True),
                         marker=dict(symbol=[data.loc[29,"Symbol"]], color=[data.loc[29,"Colour"]], line_color="black", size=[data.loc[29,'Size']], line_width=1)), row = r, col = c)
fig.update_xaxes(range = [0,6.1], dtick = 1, title = "H<sub>2</sub>O-eq (wt%)", row = r, col = c)
fig.update_yaxes(range = [0,2000], dtick = 500, title = "CO<sub>2</sub>-eq (ppm)", row = r, col = c)
fig.add_annotation(x=0.3, y=1880,text="e",showarrow=False, bordercolor = "black", bgcolor="white",
                   width=15, height=15,font=dict(color="black",size=15),textangle=0,  row = r, col = c)

# Fe3+/FeT-S
r = 3
c = 2
fig.add_trace(go.Scatter(mode = "markers", x=data["Fe3FeT"], y=data["STppm"],showlegend = False, 
                         marker=dict(symbol=data["Symbol"], color=data["Colour"], line_color="black", size=data['Size'], line_width=1)), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_1['Fe3+/FeT'], y=degassing_closed_1['ST_ppmw'], line_color = "steelblue", line_width = 3, showlegend=False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=sulfur_X['ferric_ratio'], y=sulfur_X['wS_melt'], line_color = "purple", line_width = 3, showlegend=False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=chosetto_Fe3_FeT, y=chosetto['wm_S']*1000000., line_color = "green", line_width = 3, showlegend=False,
                         line_dash = "solid"), row = r, col = c)
#fig.add_trace(go.Scatter(mode = "lines", x=evo['Fe3+/FeT'], y=evo['ST_ppmw'], line_color = "steelblue", line_width = 3, showlegend=False,
#                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "markers", x=[data.loc[29,"Fe3FeT"]], y=[data.loc[29,"STppm"]],showlegend = False, 
                         error_y=dict(type='constant', value=2.*data.loc[29,"STppm_sd"],visible=True),
                         error_x=dict(type='constant', value=2.*data.loc[29,"Fe3FeT_sd"],visible=True),
                         marker=dict(symbol=[data.loc[29,"Symbol"]], color=[data.loc[29,"Colour"]], line_color="black", size=[data.loc[29,'Size']], line_width=1)), row = r, col = c)
fig.update_xaxes(range = [0.1,0.3], dtick = 0.05, title = "Fe<sup>3+</sup>/Fe<sub>T</sub>",tickformat = ".2f", row = r, col = c)
fig.update_yaxes(range = [0,2000], dtick = 500, title = "S<sub>T</sub> (ppm)", side='right', row = r, col = c)
fig.add_annotation(x=0.11, y=1880,text="f",showarrow=False, bordercolor = "black", bgcolor="white",
                   width=15, height=15,font=dict(color="black",size=15),textangle=0,  row = r, col = c)

# H2O-S
r = 3
c = 3
fig.add_trace(go.Scatter(mode = "markers", x=data["H2O"], y=data["STppm"],showlegend = False, 
                         marker=dict(symbol=data["Symbol"], color=data["Colour"], line_color="black", size=data['Size'], line_width=1)), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_1['H2OT-eq_wtpc'], y=degassing_closed_1['ST_ppmw'], line_color = "steelblue", line_width = 3, showlegend=False, 
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=sulfur_X['wH2O_melt'], y=sulfur_X['wS_melt'], line_color = "purple", line_width = 3, showlegend=False, 
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=chosetto['wm_H2O']*100., y=chosetto['wm_S']*1000000., line_color = "green", line_width = 3, showlegend=False, 
                         line_dash = "solid"), row = r, col = c)
#fig.add_trace(go.Scatter(mode = "lines", x=evo['H2OT-eq_wtpc'], y=evo['ST_ppmw'], line_color = "steelblue", line_width = 3, showlegend=False, 
#                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "markers", x=[data.loc[29,"H2O"]], y=[data.loc[29,"STppm"]],showlegend = False, 
                         error_y=dict(type='constant', value=2.*data.loc[29,"STppm_sd"],visible=True),
                         error_x=dict(type='constant', value=2.*data.loc[29,"H2O_sd"],visible=True),
                         marker=dict(symbol=[data.loc[29,"Symbol"]], color=[data.loc[29,"Colour"]], line_color="black", size=[data.loc[29,'Size']], line_width=1)), row = r, col = c)
fig.update_xaxes(range = [0,6.1], dtick = 1, title = "H<sub>2</sub>O-eq (wt%)", row = r, col = c)
fig.update_yaxes(range = [0,2000], dtick = 500, title = "S<sub>T</sub> (ppm)",  side="right",row = r, col = c)
fig.add_annotation(x=0.3, y=1880,text="g",showarrow=False, bordercolor = "black", bgcolor="white",
                   width=15, height=15,font=dict(color="black",size=15),textangle=0,  row = r, col = c)

# create legend
fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="diamond", color="grey", line_color="black", size=10., line_width=1),
                         name="S. Mariana Trough (SPG)", legendgroup="data", legendgrouptitle_text="Data (a\u2013g)"))
fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="circle", color="grey", line_color="black", size=10., line_width=1),
                         name="Agrigan (MI)", legendgroup="data"))
fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="triangle-up", color="grey", line_color="black", size=10., line_width=1),
                         name="Sarigan (MI)", legendgroup="data"))
fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="square", color="grey", line_color="black", size=10., line_width=1),
                         name="Alamagan (MI)", legendgroup="data"))
fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="star", color="orange", line_color="black", size=10., line_width=1),
                         name="Ala02-16A (MI)", legendgroup="data"))
fig.add_trace(go.Scatter(mode = "lines", x=[-1000,-1000], y=[-1000,-1000], line_color = "darkgrey", line_width = 1, 
                         line_dash = "solid", name="isobar", legendgroup="VolFe", legendgrouptitle_text="VolFe results (b\u2013d)"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=[-1000,-1000], y=[-1000,-1000], line_color = "steelblue", line_width = 3, 
                         line_dash = "solid", name="closed/degas", legendgroup="VolFe"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=[-1000,-1000], y=[-1000,-1000], line_color = "steelblue", line_width = 3, 
                         line_dash = "dash", name="open/degas", legendgroup="VolFe"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=[-1000,-1000], y=[-1000,-1000], line_color = "steelblue", line_width = 3, 
                         line_dash = "dot", name="open/regas", legendgroup="VolFe"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=[-1000,-1000], y=[-1000,-1000], line_color = "firebrick", line_width = 3, 
                         line_dash = "solid", name="closed/degas (1 wt% CO<sub>2</sub>)", legendgroup="VolFe"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=[-1000,-1000], y=[-1000,-1000], line_color = "firebrick", line_width = 3, 
                         line_dash = "dot", name="closed/regas (1 wt% CO<sub>2</sub>)", legendgroup="VolFe"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=[-1000,-1000], y=[-1000,-1000], line_color = "steelblue", line_width = 3, 
                         line_dash = "solid", name="VolFe", legendgroup="comparison", legendgrouptitle_text="Comparison (e\u2013g)"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=[-1000,-1000], y=[-1000,-1000], line_color = "black", line_width = 3, 
                         line_dash = "solid", name="VESIcal (Dixon)", legendgroup="comparison"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=[-1000,-1000], y=[-1000,-1000], line_color = "purple", line_width = 3, 
                         line_dash = "solid", name="Sulfur_X", legendgroup="comparison"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=[-1000,-1000], y=[-1000,-1000], line_color = "green", line_width = 3, 
                         line_dash = "solid", name="CHOSETTO", legendgroup="comparison"), row = r, col = c)
fig.update_layout(legend=dict(
    orientation="h",
    yanchor="bottom",
    y=0.83,
    xanchor="left",
    x=0.34
))

fig.update_layout(height=1000, width=1000, plot_bgcolor='rgb(255,255,255)' , showlegend = True)

fig.update_xaxes(showline=True, linewidth=1, linecolor='black', mirror=True, 
                 ticks="inside", ticklen=5, title_standoff = 0, tickcolor="black",
                 title_font=dict(size=15, family='Helvetica', color='black'), 
                 tickfont=dict(family='Helvetica', color='black', size=12))
fig.update_yaxes(showline=True, linewidth=1, linecolor='black', mirror=True, 
                 ticks="inside", ticklen=5, title_standoff = 0, tickcolor="black",
                title_font=dict(size=15, family='Helvetica', color='black'), 
                 tickfont=dict(family='Helvetica', color='black', size=12))

fig.update_layout(font_family="Helvetica",font_color="black")

fig.show()

In [140]:
fig = make_subplots(rows=1, cols=3,
                   shared_yaxes = False, shared_xaxes = False,
                   vertical_spacing=0.08, horizontal_spacing=0.06)

data = pvsat

# T-P
r = 1
c = 1
fig.add_trace(go.Scatter(mode = "markers", x=data["T_C"], y=data["P_bar"], showlegend=False,
                         marker=dict(symbol=my_analyses["Symbol"], color='white', line_color="black", size=my_analyses['Size'], line_width=1)), row = r, col = c)
fig.add_trace(go.Scatter(mode = "markers", x=[pvsat_stats["T_mean"]], y=[pvsat_stats["P_mean"]], showlegend=False,
                         error_x=dict(type='constant', value=2*pvsat_stats["T_sd"],visible=True),
                         error_y=dict(type='constant', value=2*pvsat_stats["P_sd"],visible=True),
                         marker=dict(symbol=[my_analyses.loc[29,"Symbol"]], color='black', line_color="black", size=[my_analyses.loc[29,'Size']], line_width=1)), row = r, col = c)
fig.add_trace(go.Scatter(mode = "markers", x=[pvsat_stats["T_mean"]], y=[pvsat_stats["P_mean"]], showlegend=False,
                         marker=dict(symbol=[my_analyses.loc[29,"Symbol"]], color='white', line_color="black", size=[my_analyses.loc[29,'Size']], line_width=1)), row = r, col = c)
fig.update_xaxes(range = [1000,1200], dtick = 50, title = "<i>T</i> (\u00B0C)", row = r, col = c)
fig.update_yaxes(range = [4000,0], dtick = 1000, title = "<i>P<sup>v</sup></i><sub>sat</sub> (bar)",  row = r, col = c)
fig.add_annotation(x=1010, y=250,text="a",showarrow=False, bordercolor = "black", bgcolor="white",
                   width=15, height=15,font=dict(color="black",size=15),textangle=0,  row = r, col = c)

# DFMQ-P
r = 1
c = 2
fig.add_trace(go.Scatter(mode = "markers", x=data["fO2_DFMQ"], y=data["P_bar"], showlegend=False,
                         marker=dict(symbol=my_analyses["Symbol"], color='white', line_color="black", size=my_analyses['Size'], line_width=1)), row = r, col = c)
fig.add_trace(go.Scatter(mode = "markers", x=[pvsat_stats["fO2_mean"]], y=[pvsat_stats["P_mean"]], showlegend=False,
                         error_x=dict(type='constant', value=2*pvsat_stats["fO2_sd"],visible=True),
                         error_y=dict(type='constant', value=2*pvsat_stats["P_sd"],visible=True),
                         marker=dict(symbol=[my_analyses.loc[29,"Symbol"]], color='black', line_color="black", size=[my_analyses.loc[29,'Size']], line_width=1)), row = r, col = c)
fig.add_trace(go.Scatter(mode = "markers", x=[pvsat_stats["fO2_mean"]], y=[pvsat_stats["P_mean"]], showlegend=False,
                         marker=dict(symbol=[my_analyses.loc[29,"Symbol"]], color='white', line_color="black", size=[my_analyses.loc[29,'Size']], line_width=1)), row = r, col = c)
fig.update_yaxes(range = [4000,0], dtick = 1000, row = r, col = c)
fig.update_xaxes(range = [-0.5,3], dtick = 0.5, title = "log<sub>10</sub><i>f</i><sub>O<sub>2</sub></sub> (\u0394FMQ) [Fe<sup>3+</sup>/Fe<sub>T</sub>]", tickformat = ".1f", row = r, col = c)
fig.add_annotation(x=-0.3, y=250,text="b",showarrow=False, bordercolor = "black", bgcolor="white",
                   width=15, height=15,font=dict(color="black",size=15),textangle=0,  row = r, col = c)

# fO2-fO2
r = 1
c = 3
fig.add_trace(go.Scatter(mode = "lines", x=[0,4000], y=[0,4000], line_color = "darkgrey", line_width = 1, showlegend = False, 
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=[0,2], y=[-0.5,1.5], line_color = "darkgrey", line_width = 1,showlegend=False, 
                         line_dash = "dash"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=[0,2], y=[0.5,2.5], line_color = "darkgrey", line_width = 1,showlegend=False,
                         line_dash = "dash"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "markers", x=pvsat["fO2_DFMQ"], y=fO2fromS["DFMQ-sulfide"], showlegend = False,
                         marker=dict(symbol=my_analyses["Symbol"], color='white', line_color="black", size=my_analyses['Size'], line_width=1)), row = r, col = c)
fig.add_trace(go.Scatter(mode = "markers", x=[pvsat_stats["fO2_mean"]], y=[fO2fromS_stats["fO2_mean"]], showlegend = False,
                         error_x=dict(type='constant', value=2*pvsat_stats["fO2_sd"],visible=True),
                         error_y=dict(type='constant', value=2*fO2fromS_stats["fO2_sd"],visible=True),
                         marker=dict(symbol=[my_analyses.loc[29,"Symbol"]], color='black', line_color="black", size=[my_analyses.loc[29,'Size']], line_width=1)), row = r, col = c)
fig.add_trace(go.Scatter(mode = "markers", x=[pvsat_stats["fO2_mean"]], y=[fO2fromS_stats["fO2_mean"]], showlegend = False,
                         marker=dict(symbol=[my_analyses.loc[29,"Symbol"]], color='white', line_color="black", size=[my_analyses.loc[29,'Size']], line_width=1)), row = r, col = c)
fig.update_yaxes(range = [0,2], dtick = 0.5, title = "log<sub>10</sub><i>f</i><sub>O<sub>2</sub></sub> (\u0394FMQ) [measured S]", tickformat = ".1f",side='right', row = r, col = c)
fig.update_xaxes(range = [0,2], dtick = 0.5, title = "log<sub>10</sub><i>f</i><sub>O<sub>2</sub></sub> (\u0394FMQ) [Fe<sup>3+</sup>/Fe<sub>T</sub>]",  tickformat = ".1f", row = r, col = c)
fig.add_annotation(x=0.1, y=1.9,text="c",showarrow=False, bordercolor = "black", bgcolor="white",
                   width=15, height=15,font=dict(color="black",size=15),textangle=0,  row = r, col = c)

# create legend
fig.add_trace(go.Scatter(mode = "lines", x=[-1000,-1000], y=[-1000,-1000], line_color="darkgrey", line_width=1,
                         name="one-to-one"))
fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="diamond", color="white", line_color="black", size=10., line_width=1),
                         name="S. Mariana Trough (SPG) [calculated]"))
fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="circle", color="white", line_color="black", size=10., line_width=1),
                         name="Agrigan (MI) [calculated]"))
fig.add_trace(go.Scatter(mode = "lines", x=[-1000,-1000], y=[-1000,-1000], line_color="darkgrey", line_width=1,line_dash='dash',
                         name="\u00B10.5\u0394FMQ"))
fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="triangle-up", color="white", line_color="black", size=10., line_width=1),
                         name="Sarigan (MI) [calculated]"))
fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="square", color="white", line_color="black", size=10., line_width=1),
                         name="Alamagan (MI) [calculated]"))
fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="circle", color="white", line_color="white", size=10., line_width=1),
                         name=""))
fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="star", color="lightgrey", line_color="black", size=10., line_width=1),
                         name="Ala02-16A (MI) [calculated]"))
fig.update_layout(legend=dict(
    orientation="h",
    yanchor="bottom",
    y=-0.6,
    xanchor="left",
    x=0
))

fig.update_layout(height=400, width=1000, plot_bgcolor='rgb(255,255,255)' , showlegend = True)

fig.update_xaxes(showline=True, linewidth=1, linecolor='black', mirror=True, 
                 ticks="inside", ticklen=5, title_standoff = 0, tickcolor="black",
                 title_font=dict(size=15, family='Helvetica', color='black'), 
                 tickfont=dict(family='Helvetica', color='black', size=12))
fig.update_yaxes(showline=True, linewidth=1, linecolor='black', mirror=True, 
                 ticks="inside", ticklen=5, title_standoff = 0, tickcolor="black",
                title_font=dict(size=15, family='Helvetica', color='black'), 
                 tickfont=dict(family='Helvetica', color='black', size=12))

fig.update_layout(font_family="Helvetica",font_color="black")

fig.show()

In [148]:
fig = make_subplots(rows=2, cols=3,
                   shared_yaxes = True, shared_xaxes = False,
                   vertical_spacing=0.08, horizontal_spacing=0.03)

# H2O
r = 1
c = 1
value = "H2OT-eq_wtpc"
fig.add_trace(go.Scatter(mode = "markers", x=pvsat[value], y=pvsat["P_bar"], showlegend=False,
                         marker=dict(symbol=my_analyses["Symbol"], color=my_analyses["Colour"], line_color="black", size=my_analyses['Size'], line_width=1)), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_1[value], y=degassing_closed_1['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_2[value], y=degassing_closed_2['P_bar'], line_color = "firebrick", line_width = 3, showlegend = False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_closed_2[value], y=regassing_closed_2['P_bar'], line_color = "firebrick", line_width = 3, showlegend = False,
                         line_dash = "dot"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_open[value], y=degassing_open['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "dash"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_open[value], y=regassing_open['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "dot"), row = r, col = c)
fig.update_xaxes(range = [0,6.1], dtick = 1, title = "H<sub>2</sub>O-eq (wt%)", row = r, col = c)
fig.update_yaxes(range = [5000.,0.], dtick = 1000, title = "<i>P</i> (bar)", row = r, col = c)
fig.add_annotation(x=5.7, y=300.,text="a",showarrow=False, bordercolor = "black", bgcolor="white",
                   width=15, height=15,font=dict(color="black",size=15),textangle=0,  row = r, col = c)

# CO2
r = 1
c = 2
value = "CO2T-eq_ppmw"
fig.add_trace(go.Scatter(mode = "markers", x=pvsat[value], y=pvsat["P_bar"], showlegend=False,
                         marker=dict(symbol=my_analyses["Symbol"], color=my_analyses["Colour"], line_color="black", size=my_analyses['Size'], line_width=1)), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_1[value], y=degassing_closed_1['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_2[value], y=degassing_closed_2['P_bar'], line_color = "firebrick", line_width = 3, showlegend = False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_closed_2[value], y=regassing_closed_2['P_bar'], line_color = "firebrick", line_width = 3, showlegend=False,
                         line_dash = "dot"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_open[value], y=degassing_open['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "dash"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_open[value], y=regassing_open['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "dot"), row = r, col = c)
fig.update_xaxes(range = [0,2500], dtick = 500, title = "CO<sub>2</sub>-eq (ppmw)", row = r, col = c)
fig.add_annotation(x=2350, y=300.,text="b",showarrow=False, bordercolor = "black", bgcolor="white",
                   width=15, height=15,font=dict(color="black",size=15),textangle=0,  row = r, col = c)

# S
r = 1
c = 3
value = "ST_ppmw"
fig.add_trace(go.Scatter(mode = "markers", x=pvsat[value], y=pvsat["P_bar"], showlegend=False,
                         marker=dict(symbol=my_analyses["Symbol"], color=my_analyses["Colour"], line_color="black", size=my_analyses['Size'], line_width=1)), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_1[value], y=degassing_closed_1['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_2[value], y=degassing_closed_2['P_bar'], line_color = "firebrick", line_width = 3, showlegend = False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_closed_2[value], y=regassing_closed_2['P_bar'], line_color = "firebrick", line_width = 3, showlegend=False,
                         line_dash = "dot"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_open[value], y=degassing_open['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "dash"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_open[value], y=regassing_open['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "dot"), row = r, col = c)
fig.update_xaxes(range = [0,2000], dtick = 500, title = "S<sub>T</sub> (ppmw)", row = r, col = c)
fig.add_annotation(x=1900, y=300.,text="c",showarrow=False, bordercolor = "black", bgcolor="white",
                   width=15, height=15,font=dict(color="black",size=15),textangle=0,  row = r, col = c)

# Fe3+/FeT
r = 2
c = 1
value = "Fe3+/FeT"
fig.add_trace(go.Scatter(mode = "markers", x=pvsat[value], y=pvsat["P_bar"], showlegend=False,
                         marker=dict(symbol=my_analyses["Symbol"], color=my_analyses["Colour"], line_color="black", size=my_analyses['Size'], line_width=1)), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_1[value], y=degassing_closed_1['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_2[value], y=degassing_closed_2['P_bar'], line_color = "firebrick", line_width = 3, showlegend = False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_closed_2[value], y=regassing_closed_2['P_bar'], line_color = "firebrick", line_width = 4, showlegend=False,
                         line_dash = "dot"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_open[value], y=degassing_open['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "dash"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_open[value], y=regassing_open['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "dot"), row = r, col = c)
fig.update_xaxes(range = [0.1,0.3], dtick = 0.05, title = "Fe<sup>3+</sup>/Fe<sub>T</sub>",tickformat = ".2f", row = r, col = c)
fig.update_yaxes(range = [5000.,0.], dtick = 1000, title = "<i>P</i> (bar)", row = r, col = c)
fig.add_annotation(x=0.29, y=300.,text="d",showarrow=False, bordercolor = "black", bgcolor="white",
                   width=15, height=15,font=dict(color="black",size=15),textangle=0,  row = r, col = c)

# S6+/ST
r = 2
c = 2
value = "S6+/ST"
fig.add_trace(go.Scatter(mode = "markers", x=pvsat[value], y=pvsat["P_bar"], showlegend=False,
                         marker=dict(symbol=my_analyses["Symbol"], color='white', line_color="black", size=my_analyses['Size'], line_width=1)), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_1[value], y=degassing_closed_1['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_2[value], y=degassing_closed_2['P_bar'], line_color = "firebrick", line_width = 3, showlegend = False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_closed_2[value], y=regassing_closed_2['P_bar'], line_color = "firebrick", line_width = 3, showlegend=False,
                         line_dash = "dot"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_open[value], y=degassing_open['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "dash"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_open[value], y=regassing_open['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "dot"), row = r, col = c)
fig.update_xaxes(range = [0,1], dtick = 0.2, title = "S<sup>6+</sup>/S<sub>T</sub>",tickformat = ".1f", row = r, col = c)
fig.add_annotation(x=0.95, y=300.,text="e",showarrow=False, bordercolor = "black", bgcolor="white",
                   width=15, height=15,font=dict(color="black",size=15),textangle=0,  row = r, col = c)

# DFMQ
r = 2
c = 3
value = "fO2_DFMQ"
fig.add_trace(go.Scatter(mode = "markers", x=pvsat[value], y=pvsat["P_bar"], showlegend=False,
                         marker=dict(symbol=my_analyses["Symbol"], color='white', line_color="black", size=my_analyses['Size'], line_width=1)), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_1[value], y=degassing_closed_1['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_2[value], y=degassing_closed_2['P_bar'], line_color = "firebrick", line_width = 3, showlegend = False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_closed_2[value], y=regassing_closed_2['P_bar'], line_color = "firebrick", line_width = 4, showlegend=False,
                         line_dash = "dot"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_open[value], y=degassing_open['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "dash"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_open[value], y=regassing_open['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "dot"), row = r, col = c)
fig.update_xaxes(range = [0,2], dtick = 0.5, title = "log<sub>10</sub><i>f</i><sub>O<sub>2</sub></sub> (\u0394FMQ)",tickformat = ".1f", row = r, col = c)
fig.add_annotation(x=1.9, y=300.,text="f",showarrow=False, bordercolor = "black", bgcolor="white",
                   width=15, height=15,font=dict(color="black",size=15),textangle=0,  row = r, col = c)

# create legend
fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="diamond", color="grey", line_color="black", size=10., line_width=1),
                         name="S. Mariana Trough (SPG)",legendgroup="measured", legendgrouptitle_text="Measured"))
fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="diamond", color="white", line_color="black", size=10., line_width=1),
                         name="S. Mariana Trough (SPG)", legendgroup="calculated", legendgrouptitle_text="Calculated"))
fig.add_trace(go.Scatter(mode = "lines", x=[-1000,-1000], y=[-1000,-1000], line_color = "steelblue", line_width = 3, 
                         line_dash = "solid", name="closed/degas", legendgroup="bulk", legendgrouptitle_text="Initial CO<sub>2</sub>-eq = 999 ppmw"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=[-1000,-1000], y=[-1000,-1000], line_color = "firebrick", line_width = 3, 
                         line_dash = "solid", name="closed/degas", legendgroup="mv", legendgrouptitle_text="Initial CO<sub>2</sub>-eq = 1 wt%"), row = r, col = c)

fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="circle", color="grey", line_color="black", size=10., line_width=1),
                         name="Agrigan (MI)",legendgroup="measured"))
fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="circle", color="white", line_color="black", size=10., line_width=1),
                         name="Agrigan (MI)",legendgroup="calculated"))
fig.add_trace(go.Scatter(mode = "lines", x=[-1000,-1000], y=[-1000,-1000], line_color = "steelblue", line_width = 3, 
                         line_dash = "dash", name="open/degas", legendgroup="bulk"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=[-1000,-1000], y=[-1000,-1000], line_color = "firebrick", line_width = 3, 
                         line_dash = "dot", name="closed/regas", legendgroup="mv"), row = r, col = c)

fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="triangle-up", color="grey", line_color="black", size=10., line_width=1),
                         name="Sarigan (MI)",legendgroup="measured"))
fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="triangle-up", color="white", line_color="black", size=10., line_width=1),
                         name="Sarigan (MI)",legendgroup="calculated"))
fig.add_trace(go.Scatter(mode = "lines", x=[-1000,-1000], y=[-1000,-1000], line_color = "steelblue", line_width = 3, 
                         line_dash = "dot", name="open/regas", legendgroup="bulk"), row = r, col = c)


fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="square", color="grey", line_color="black", size=10., line_width=1),
                         name="Alamagan (MI)",legendgroup="measured"))
fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="square", color="white", line_color="black", size=10., line_width=1),
                         name="Alamagan (MI)",legendgroup="calculated"))

fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="star", color="orange", line_color="black", size=10., line_width=1),
                         name="Ala02-16A (MI)",legendgroup="measured"))
fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="star", color="white", line_color="black", size=10., line_width=1),
                         name="Ala02-16A (MI)",legendgroup="calculated"))





fig.update_layout(legend=dict(
    orientation="h",
    yanchor="bottom",
    y=-0.35,
    xanchor="left",
    x=0
))

fig.update_layout(height=800, width=1000, plot_bgcolor='rgb(255,255,255)' , showlegend = True)

fig.update_xaxes(showline=True, linewidth=1, linecolor='black', mirror=True, 
                 ticks="inside", ticklen=5, title_standoff = 0, tickcolor="black",
                 title_font=dict(size=15, family='Helvetica', color='black'), 
                 tickfont=dict(family='Helvetica', color='black', size=12))
fig.update_yaxes(showline=True, linewidth=1, linecolor='black', mirror=True, 
                 ticks="inside", ticklen=5, title_standoff = 0, tickcolor="black",
                title_font=dict(size=15, family='Helvetica', color='black'), 
                 tickfont=dict(family='Helvetica', color='black', size=12))

fig.update_layout(font_family="Helvetica",font_color="black")

fig.show()

In [152]:
fig = make_subplots(rows=2, cols=3,
                   shared_yaxes = True, shared_xaxes = False,
                   vertical_spacing=0.08, horizontal_spacing=0.03)

# H2O vapor
r = 1
c = 1
value = "xgH2O_mf"
fig.add_trace(go.Scatter(mode = "markers", x=pvsat[value], y=pvsat["P_bar"], showlegend=False,
                         marker=dict(symbol=my_analyses["Symbol"], color='white', line_color="black", size=my_analyses['Size'], line_width=1)), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_1[value], y=degassing_closed_1['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_2[value], y=degassing_closed_2['P_bar'], line_color = "firebrick", line_width = 3, showlegend = False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_closed_2[value], y=regassing_closed_2['P_bar'], line_color = "firebrick", line_width = 3, showlegend = False,
                         line_dash = "dot"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_open[value], y=degassing_open['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "dash"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_open[value], y=regassing_open['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "dot"), row = r, col = c)
fig.update_xaxes(range = [0,1], dtick = 0.2, title = "<i>x<sup>v</sup></i><sub>H<sub>2</sub>O</sub> (mole fraction)",tickformat = ".1f", row = r, col = c)
fig.update_yaxes(range = [5000.,0.], dtick = 1000, title = "<i>P</i> (bar)", row = r, col = c)
fig.add_annotation(x=0.05, y=300.,text="a",showarrow=False, bordercolor = "black", bgcolor="white",
                   width=15, height=15,font=dict(color="black",size=15),textangle=0,  row = r, col = c)

# CO2 vapor
r = 1
c = 2
value = "xgCO2_mf"
fig.add_trace(go.Scatter(mode = "markers", x=pvsat[value], y=pvsat["P_bar"], showlegend=False,
                         marker=dict(symbol=my_analyses["Symbol"], color='white', line_color="black", size=my_analyses['Size'], line_width=1)), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_1[value], y=degassing_closed_1['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_2[value], y=degassing_closed_2['P_bar'], line_color = "firebrick", line_width = 3, showlegend = False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_closed_2[value], y=regassing_closed_2['P_bar'], line_color = "firebrick", line_width = 3, showlegend=False,
                         line_dash = "dot"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_open[value], y=degassing_open['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "dash"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_open[value], y=regassing_open['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "dot"), row = r, col = c)
fig.update_xaxes(range = [0,1], dtick = 0.2, title = "<i>x<sup>v</sup></i><sub>CO<sub>2</sub></sub> (mole fraction)",tickformat = ".1f", row = r, col = c)
fig.add_annotation(x=0.95, y=300.,text="b",showarrow=False, bordercolor = "black", bgcolor="white",
                   width=15, height=15,font=dict(color="black",size=15),textangle=0,  row = r, col = c)

# SO2 vapor
r = 2
c = 1
value = "xgSO2_mf"
fig.add_trace(go.Scatter(mode = "markers", x=pvsat[value], y=pvsat["P_bar"], showlegend=False,
                         marker=dict(symbol=my_analyses["Symbol"], color='white', line_color="black", size=my_analyses['Size'], line_width=1)), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_1[value], y=degassing_closed_1['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_2[value], y=degassing_closed_2['P_bar'], line_color = "firebrick", line_width = 3, showlegend = False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_closed_2[value], y=regassing_closed_2['P_bar'], line_color = "firebrick", line_width = 3, showlegend=False,
                         line_dash = "dot"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_open[value], y=degassing_open['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "dash"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_open[value], y=regassing_open['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "dot"), row = r, col = c)
fig.update_xaxes(range = [0,0.1], dtick = 0.02, title = "<i>x<sup>v</sup></i><sub>SO<sub>2</sub></sub> (mole fraction)",tickformat = ".2f", row = r, col = c)
fig.update_yaxes(range = [5000.,0.], dtick = 1000, title = "<i>P</i> (bar)", row = r, col = c)
fig.add_annotation(x=0.095, y=300.,text="c",showarrow=False, bordercolor = "black", bgcolor="white",
                   width=15, height=15,font=dict(color="black",size=15),textangle=0,  row = r, col = c)

# H2S vapor
r = 2
c = 2
value = "xgH2S_mf"
fig.add_trace(go.Scatter(mode = "markers", x=pvsat[value], y=pvsat["P_bar"], showlegend=False,
                         marker=dict(symbol=my_analyses["Symbol"], color='white', line_color="black", size=my_analyses['Size'], line_width=1)), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_1[value], y=degassing_closed_1['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_2[value], y=degassing_closed_2['P_bar'], line_color = "firebrick", line_width = 3, showlegend = False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_closed_2[value], y=regassing_closed_2['P_bar'], line_color = "firebrick", line_width = 3, showlegend=False,
                         line_dash = "dot"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_open[value], y=degassing_open['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "dash"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_open[value], y=regassing_open['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "dot"), row = r, col = c)
fig.update_xaxes(range = [0,0.1], dtick = 0.02, title = "<i>x<sup>v</sup></i><sub>H<sub>2</sub>S</sub> (mole fraction)",tickformat = ".2f", row = r, col = c)
fig.add_annotation(x=0.095, y=300.,text="d",showarrow=False, bordercolor = "black", bgcolor="white",
                   width=15, height=15,font=dict(color="black",size=15),textangle=0,  row = r, col = c)

# CS
r = 2
c = 3
value = "xgC_S_mf"
fig.add_trace(go.Scatter(mode = "markers", x=pvsat[value], y=pvsat["P_bar"], showlegend=False,
                         marker=dict(symbol=my_analyses["Symbol"], color='white', line_color="black", size=my_analyses['Size'], line_width=1)), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_1[value], y=degassing_closed_1['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_1[value], y=degassing_closed_1['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_closed_2[value], y=degassing_closed_2['P_bar'], line_color = "firebrick", line_width = 3, showlegend = False,
                         line_dash = "solid"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_closed_2[value], y=regassing_closed_2['P_bar'], line_color = "firebrick", line_width = 3, showlegend=False,
                         line_dash = "dot"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=degassing_open[value], y=degassing_open['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "dash"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=regassing_open[value], y=regassing_open['P_bar'], line_color = "steelblue", line_width = 3, showlegend = False,
                         line_dash = "dot"), row = r, col = c)
fig.update_xaxes(range = [0,50], dtick = 10, title = "C<sub>T</sub>/S<sub>T</sub> (mole fraction)", row = r, col = c)
fig.add_annotation(x=47, y=300.,text="e",showarrow=False, bordercolor = "black", bgcolor="white",
                   width=15, height=15,font=dict(color="black",size=15),textangle=0,  row = r, col = c)

# create legend
fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="diamond", color='white', line_color="black", size=10., line_width=1),
                         name="S. Mariana Trough (SPG)", legendgroup='calc', legendgrouptitle_text="Calculated"))
fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="circle", color='white', line_color="black", size=10., line_width=1),
                         name="Agrigan (MI)", legendgroup='calc'))
fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="triangle-up", color='white', line_color="black", size=10., line_width=1),
                         name="Sarigan (MI)",legendgroup='calc'))
fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="square", color='white', line_color="black", size=10., line_width=1),
                         name="Alamagan (MI)",legendgroup='calc'))
fig.add_trace(go.Scatter(mode = "markers", x=[-1000,-1000], y=[-1000,-1000],
                         marker=dict(symbol="star", color='white', line_color="black", size=10., line_width=1),
                         name="Ala02-16A (MI)",legendgroup='calc'))
fig.add_trace(go.Scatter(mode = "lines", x=[-1000,-1000], y=[-1000,-1000], line_color = "steelblue", line_width = 3, 
                         line_dash = "solid", name="closed/degas",legendgroup='melt',legendgrouptitle_text="Initial CO<sub>2</sub>-eq = 999 ppmw"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=[-1000,-1000], y=[-1000,-1000], line_color = "firebrick", line_width = 3, 
                         line_dash = "solid", name="closed/degas",legendgroup='melt+v',legendgrouptitle_text="Initial CO<sub>2</sub>-eq = 1 wt%"), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=[-1000,-1000], y=[-1000,-1000], line_color = "firebrick", line_width = 3, 
                         line_dash = "dot", name="closed/regas",legendgroup='melt+v'), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=[-1000,-1000], y=[-1000,-1000], line_color = "steelblue", line_width = 3, 
                         line_dash = "dash", name="open/degas",legendgroup='melt'), row = r, col = c)
fig.add_trace(go.Scatter(mode = "lines", x=[-1000,-1000], y=[-1000,-1000], line_color = "steelblue", line_width = 3, 
                         line_dash = "dot", name="open/regas",legendgroup='melt'), row = r, col = c)
fig.update_layout(legend=dict(
    #orientation="h",
    yanchor="bottom",
    y=0.475,
    xanchor="left",
    x=0.7
))

fig.update_layout(height=700, width=1000, plot_bgcolor='rgb(255,255,255)' , showlegend = True)

fig.update_xaxes(showline=True, linewidth=1, linecolor='black', mirror=True, 
                 ticks="inside", ticklen=5, title_standoff = 0, tickcolor="black",
                 title_font=dict(size=15, family='Helvetica', color='black'), 
                 tickfont=dict(family='Helvetica', color='black', size=12))
fig.update_yaxes(showline=True, linewidth=1, linecolor='black', mirror=True, 
                 ticks="inside", ticklen=5, title_standoff = 0, tickcolor="black",
                title_font=dict(size=15, family='Helvetica', color='black'), 
                 tickfont=dict(family='Helvetica', color='black', size=12))

fig.update_layout(font_family="Helvetica",font_color="black")

fig.show()

## Saving outputs

Saving all outputs to csv.

In [27]:
comp_error.to_csv('../files/marianas_comp_error.csv', index=False, header=True)
pvsat.to_csv('../files/marianas_pvsat.csv', index=False, header=True)
pvsat_error.to_csv('../files/marianas_pvsat_error.csv', index=False, header=True)
fO2fromS.to_csv('../files/marianas_fO2fromS.csv', index=False, header=True)
fO2fromS_error.to_csv('../files/marianas_fO2fromS_error.csv', index=False, header=True)
degassing_closed_1.to_csv('../files/marianas_degas_closed_1.csv', index=False, header=True)
degassing_closed_2.to_csv('../files/marianas_degas_closed_2.csv', index=False, header=True)
regassing_closed_2.to_csv('../files/marianas_regas_closed_2.csv', index=False, header=True)
degassing_open.to_csv('../files/marianas_degas_open.csv', index=False, header=True)
regassing_open.to_csv('../files/marianas_regas_open.csv', index=False, header=True)
isobars.to_csv('../files/marianas_isobars.csv', index=False, header=True)

OSError: Cannot save file into a non-existent directory: '..\files'